In [9]:
!pip install -q xarray s3fs fsspec netCDF4 h5netcdf pyproj pandas numpy matplotlib pillow tqdm

In [19]:
!pip install h5py h5netcdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.1 MB/s  0:00:00


In [1]:
import os
import re
import json
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import s3fs
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from pyproj import Proj

In [2]:
# Storm case: Hurricane Laura approaching / making landfall
SATELLITE_BUCKET = "noaa-goes16"       # GOES-East during 2020
PRODUCT = "ABI-L2-MCMIPC"              # Multi-channel CONUS imagery

START_TIME = dt.datetime(2020, 8, 26, 18)  # Aug 26, 2020 18:00 UTC
END_TIME   = dt.datetime(2020, 8, 27, 12)  # Aug 27, 2020 12:00 UTC

# Approximate bounding box around Louisiana / Gulf Coast
# lon_min, lon_max, lat_min, lat_max
BBOX = (-98, -87, 24, 34)

OUT_DIR = Path("goes_project3_output")
FRAME_DIR = OUT_DIR / "frames"

OUT_DIR.mkdir(exist_ok=True)
FRAME_DIR.mkdir(exist_ok=True)

In [3]:
fs = s3fs.S3FileSystem(anon=True)

In [4]:
def goes_s3_hour_path(bucket, product, timestamp):
    year = timestamp.year
    day_of_year = timestamp.timetuple().tm_yday
    hour = timestamp.hour
    
    return f"{bucket}/{product}/{year}/{day_of_year:03d}/{hour:02d}/*.nc"


def list_goes_files(bucket, product, start_time, end_time):
    files = []
    current = start_time.replace(minute=0, second=0, microsecond=0)
    
    while current <= end_time:
        pattern = goes_s3_hour_path(bucket, product, current)
        hour_files = fs.glob(pattern)
        files.extend(hour_files)
        current += dt.timedelta(hours=1)
    
    files = sorted(files)
    return files

In [5]:
files = list_goes_files(SATELLITE_BUCKET, PRODUCT, START_TIME, END_TIME)

print(f"Found {len(files)} GOES files")
print(files[:3])
print(files[-3:])

Found 228 GOES files
['noaa-goes16/ABI-L2-MCMIPC/2020/239/18/OR_ABI-L2-MCMIPC-M6_G16_s20202391801173_e20202391803552_c20202391804102.nc', 'noaa-goes16/ABI-L2-MCMIPC/2020/239/18/OR_ABI-L2-MCMIPC-M6_G16_s20202391806173_e20202391808552_c20202391809094.nc', 'noaa-goes16/ABI-L2-MCMIPC/2020/239/18/OR_ABI-L2-MCMIPC-M6_G16_s20202391811173_e20202391813558_c20202391814100.nc']
['noaa-goes16/ABI-L2-MCMIPC/2020/240/12/OR_ABI-L2-MCMIPC-M6_G16_s20202401246171_e20202401248550_c20202401249102.nc', 'noaa-goes16/ABI-L2-MCMIPC/2020/240/12/OR_ABI-L2-MCMIPC-M6_G16_s20202401251171_e20202401253544_c20202401254093.nc', 'noaa-goes16/ABI-L2-MCMIPC/2020/240/12/OR_ABI-L2-MCMIPC-M6_G16_s20202401256171_e20202401258544_c20202401259091.nc']


In [6]:
# Optional: reduce file count for a lighter project
files_sampled = files[::2]

print(f"Using {len(files_sampled)} sampled files")

Using 114 sampled files


In [ ]:
file_objects = [fs.open(f) for f in files_sampled]

ds = xr.open_mfdataset(
    file_objects,
    combine="nested",
    concat_dim="time",
    engine="h5netcdf",
    chunks={"time": 1}
)

ds

In [ ]:
list(ds.data_vars)

In [ ]:
CHANNELS = {
    "visible_cloud_structure": "CMI_C02",
    "water_vapor_upper": "CMI_C08",
    "water_vapor_mid": "CMI_C09",
    "water_vapor_low": "CMI_C10",
    "infrared_cloud_top_temp": "CMI_C13"
}

In [ ]:
x_slice, y_slice = latlon_bbox_to_xy_slices(ds, BBOX)

ds_sub = ds.sel(x=x_slice, y=y_slice)

ds_sub

In [ ]:
needed_vars = list(CHANNELS.values())

ds_small = ds_sub[needed_vars].load()

ds_small

In [ ]:
def compute_storm_metrics(ds_small):
    rows = []
    
    for i in range(ds_small.sizes["time"]):
        frame = ds_small.isel(time=i)
        
        c13 = frame["CMI_C13"].values  # infrared brightness temperature, usually Kelvin
        
        # Remove invalid values
        c13_valid = c13[np.isfinite(c13)]
        
        if len(c13_valid) == 0:
            continue
        
        # Cold cloud threshold.
        # Lower brightness temperature usually means higher/colder cloud tops.
        cold_threshold_k = 220
        
        min_temp = float(np.nanmin(c13_valid))
        mean_temp = float(np.nanmean(c13_valid))
        cold_cloud_fraction = float(np.mean(c13_valid < cold_threshold_k))
        
        rows.append({
            "frame": i,
            "time": str(pd.to_datetime(ds_small.time.values[i])),
            "min_cloud_top_temp_k": min_temp,
            "mean_cloud_top_temp_k": mean_temp,
            "cold_cloud_fraction": cold_cloud_fraction,
            "intensity_proxy": cold_cloud_fraction * (260 - min_temp)
        })
    
    return pd.DataFrame(rows)


metrics_df = compute_storm_metrics(ds_small)
metrics_df.head()

In [ ]:
metrics_df.to_csv(OUT_DIR / "storm_metrics.csv", index=False)
metrics_df

In [ ]:
def normalize_array(arr, vmin=None, vmax=None, invert=False):
    arr = np.array(arr, dtype=float)
    
    if vmin is None:
        vmin = np.nanpercentile(arr, 2)
    if vmax is None:
        vmax = np.nanpercentile(arr, 98)
    
    norm = (arr - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)
    
    if invert:
        norm = 1 - norm
    
    return (norm * 255).astype(np.uint8)

In [ ]:
def save_grayscale_frames(ds_small, var_name, output_folder, prefix, vmin=None, vmax=None, invert=False):
    output_folder = Path(output_folder)
    output_folder.mkdir(exist_ok=True, parents=True)
    
    frame_records = []
    
    for i in tqdm(range(ds_small.sizes["time"])):
        arr = ds_small[var_name].isel(time=i).values
        
        img_arr = normalize_array(arr, vmin=vmin, vmax=vmax, invert=invert)
        img = Image.fromarray(img_arr)
        
        filename = f"{prefix}_{i:03d}.png"
        filepath = output_folder / filename
        img.save(filepath)
        
        frame_records.append({
            "frame": i,
            "time": str(pd.to_datetime(ds_small.time.values[i])),
            "variable": var_name,
            "file": f"frames/{filename}"
        })
    
    return pd.DataFrame(frame_records)

In [ ]:
c13_frames = save_grayscale_frames(
    ds_small,
    var_name="CMI_C13",
    output_folder=FRAME_DIR,
    prefix="c13_cloud_top_temp",
    vmin=190,
    vmax=300,
    invert=True
)

c13_frames.head()

In [ ]:
wv_frames = save_grayscale_frames(
    ds_small,
    var_name="CMI_C09",
    output_folder=FRAME_DIR,
    prefix="c09_water_vapor",
    vmin=None,
    vmax=None,
    invert=True
)

wv_frames.head()

In [ ]:
vis_frames = save_grayscale_frames(
    ds_small,
    var_name="CMI_C02",
    output_folder=FRAME_DIR,
    prefix="c02_visible",
    vmin=None,
    vmax=None,
    invert=False
)

vis_frames.head()

In [ ]:
frames_df = pd.concat([c13_frames, wv_frames, vis_frames], ignore_index=True)

frames_df.to_csv(OUT_DIR / "frames.csv", index=False)

frames_df.head()

In [ ]:
project_data = {
    "title": "How Do Storms Develop and Intensify?",
    "storm": "Hurricane Laura",
    "time_range": {
        "start": START_TIME.isoformat(),
        "end": END_TIME.isoformat()
    },
    "bbox": {
        "lon_min": BBOX[0],
        "lon_max": BBOX[1],
        "lat_min": BBOX[2],
        "lat_max": BBOX[3]
    },
    "variables": {
        "CMI_C13": "Infrared cloud-top temperature proxy",
        "CMI_C09": "Mid-level water vapor / atmospheric structure proxy",
        "CMI_C02": "Visible cloud structure proxy"
    },
    "frames": frames_df.to_dict(orient="records"),
    "metrics": metrics_df.to_dict(orient="records")
}

with open(OUT_DIR / "project_data.json", "w") as f:
    json.dump(project_data, f, indent=2)